DDL操作

In [2]:
from pymilvus import MilvusClient

#操作客户端
client=MilvusClient("http://localhost:19530")

#列出所有数据库
existed_databases=client.list_databases()
print(existed_databases)


['default', 'rag_demo']


In [8]:
#创建新的数据库
db_name="rag_demo"
if db_name not in existed_databases:
    client.create_database(db_name=db_name)

In [6]:
#删除数据库
client.drop_database(db_name=db_name)

collection相关操作

In [10]:
#切换数据库
client.use_database(db_name=db_name)

In [20]:
#查看数据库下的collections,相当于表tables
collections=client.list_collections(db_name=db_name)
print(collections)


[]


In [17]:
#创建collections
collection_name="docs"
client.create_collection(
    collection_name=collection_name,
    dimension=1024,
    metric_type="COSINE"
)


In [19]:
#删除collection
client.drop_collection(collection_name=collection_name)

DML操作

In [3]:
#嵌入模型初始化
from dotenv import load_dotenv#读取.env配置文件中的密钥
from langchain.embeddings.base import init_embeddings
import os # 读取环境变量

load_dotenv(override=True)# 读取.env配置文件中的密钥
DASHSCOPE_BASE_URL=os.getenv("DASHSCOPE_BASE_URL")
DASHSCOPE_API_KEY=os.getenv("DASHSCOPE_API_KEY")
embedding_model=init_embeddings(
    model="openai:qwen3.7-text-embedding",
    api_key=DASHSCOPE_API_KEY,
    base_url=DASHSCOPE_BASE_URL,
    check_embedding_ctx_length=False,#关掉 token id 预处理，直接传字符串
    chunk_size=10#适配百炼单次最多 10 条文本的限制
)


#创建collections
collection_name="docs"
client.create_collection(
    collection_name=collection_name,
    dimension=1024,
    metric_type="COSINE"
)

#查看collection的元数据
from rich import print as rprint
metadata=client.describe_collection(collection_name=collection_name)
rprint(metadata)


{
    'collection_name': 'docs',
    'auto_id': False,
    'num_shards': 1,
    'description': '',
    'fields': [
        {
            'field_id': 100,
            'name': 'id',
            'description': '',
            'type': <DataType.INT64: 5>,
            'params': {},
            'is_primary': True
        },
        {
            'field_id': 101,
            'name': 'vector',
            'description': '',
            'type': <DataType.FLOAT_VECTOR: 101>,
            'params': {'dim': 1024}
        }
    ],
    'functions': [],
    'aliases': [],
    'collection_id': 470342332038020833,
    'consistency_level': 2,
    'properties': {'timezone': 'UTC', 'namespace.sharding.enabled': 'false', 'max_field_id': '102'},
    'num_partitions': 1,
    'enable_dynamic_field': True,
    'enable_namespace': False,
    'created_timestamp': 470343895655645201,
    'update_timestamp': 470343895655645201
}

In [4]:
#准备数据
texts=["你好，我是小明",
      "我准备学一学python",
      "我觉得昆明是一个很美的地方",
      "我最想去的地方是大理"]


In [5]:
#生成嵌入向量
vectors=embedding_model.embed_documents(texts)

#查看生成的嵌入向量
print(len(vectors))
print(len(vectors[0]))
print(vectors[0][:5])

4
1024
[0.01515960693359375, 0.01678466796875, 0.0261077880859375, 0.0067291259765625, 0.0301513671875]


In [6]:
#封装为可以插入的数据格式
data=[
    {
        "id":i,
        "vector":vectors[i],
        "text":texts[i],
        "source":"demo"
    }   for i in range(len(vectors))
]

In [7]:
#写入数据
insert_res=client.upsert(
    collection_name=collection_name,
    data=data
)
print(insert_res)

{'upsert_count': 4, 'ids': [0, 1, 2, 3]}


In [9]:
#手动flush
#因为milvus不会第一时间将数据落盘，如果要看到写入效果，必须手动flush落盘，将数据刷新在磁盘中
client.flush(collection_name=collection_name)

#查看collection的统计信息
stats=client.get_collection_stats(collection_name=collection_name)
print(stats)


{'row_count': 4}


DQL操作

In [11]:
iterator=client.query_iterator(
    collection_name=collection_name,
    filter="",
    output_fields=["*"]
)
i=0
while True:
    rows=iterator.next()
    if not rows:
        break
    for row in rows:
        print(f"第{i+1}条数据",row["id"],row["text"],row["source"],row["vector"][:5])
        i+=1



第1条数据 0 你好，我是小明 demo [0.01515960693359375, 0.01678466796875, 0.0261077880859375, 0.0067291259765625, 0.0301513671875]
第2条数据 1 我准备学一学python demo [0.0188140869140625, -0.0104522705078125, 0.01195526123046875, -0.015472412109375, 0.0221099853515625]
第3条数据 2 我觉得昆明是一个很美的地方 demo [0.01457977294921875, 0.0421142578125, -0.03753662109375, -0.00235748291015625, -0.0108184814453125]
第4条数据 3 我最想去的地方是大理 demo [0.024993896484375, 0.03582763671875, -0.00652313232421875, 0.002422332763671875, 0.036895751953125]


In [13]:
#通过主键查询数据
res=client.get(
    collection_name=collection_name,
    ids=[0,1,2]
)
for i in range(len(res)):
    print(f"第{i+1}条数据",res[i]["id"],res[i]["text"],res[i]["source"],res[i]["vector"][:5])


第1条数据 0 你好，我是小明 demo [0.01515960693359375, 0.01678466796875, 0.0261077880859375, 0.0067291259765625, 0.0301513671875]
第2条数据 1 我准备学一学python demo [0.0188140869140625, -0.0104522705078125, 0.01195526123046875, -0.015472412109375, 0.0221099853515625]
第3条数据 2 我觉得昆明是一个很美的地方 demo [0.01457977294921875, 0.0421142578125, -0.03753662109375, -0.00235748291015625, -0.0108184814453125]


相似度检索

In [20]:
#查询的问题
query_text1="我叫什么名字啊"
query_vector1=embedding_model.embed_query(query_text1)
query_text2="哪里是个不错的地方呢"
query_vector2=embedding_model.embed_query(query_text2)



#进行向量数据库的检索
res=client.search(
    collection_name=collection_name,
    data=[query_vector1,query_vector2],
    limit=3,
    output_fields=["text","source"]
)
for i in res[0]:
    print(i)
for i in res[1]:
    print(i)






{'id': 0, 'distance': 0.5934134125709534, 'entity': {'text': '你好，我是小明', 'source': 'demo'}}
{'id': 1, 'distance': 0.3668643832206726, 'entity': {'text': '我准备学一学python', 'source': 'demo'}}
{'id': 3, 'distance': 0.33965539932250977, 'entity': {'text': '我最想去的地方是大理', 'source': 'demo'}}
{'id': 3, 'distance': 0.5292976498603821, 'entity': {'text': '我最想去的地方是大理', 'source': 'demo'}}
{'id': 2, 'distance': 0.5279907584190369, 'entity': {'text': '我觉得昆明是一个很美的地方', 'source': 'demo'}}
{'id': 0, 'distance': 0.39411813020706177, 'entity': {'text': '你好，我是小明', 'source': 'demo'}}
